## Model Context Protocol

Build a small MCP server, inspect what it puts on the wire, call it from a client,
and hand its tools to a LangChain agent.

The point of this notebook is that the server is written once and consumed three
different ways without changing a line of it.


### Install

The course environment ships MCP SDK v1. This module needs v2, which implements the 2026-07-28 specification.

In [ ]:
!pip install --quiet "mcp[cli]>=2"

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

In [ ]:
import importlib.metadata as md
print("mcp SDK:", md.version("mcp"))     # must be 2.x

### 1. The server

Type hints become the JSON Schema. The docstring becomes the description the model
reads when deciding whether to call the tool. Nothing else is declared.

In [ ]:
%%writefile meter_server.py
from mcp.server import MCPServer

mcp = MCPServer("meters", version="1.0.0")


@mcp.tool()
def find_meter(address: str) -> str:
    """Return the electricity meter ID for a delivery address."""
    return {"Kerkstraat 12": "MTR-8801",
            "Dorpsweg 5": "MTR-9042"}.get(address, "UNKNOWN")


@mcp.tool()
def read_meter(meter_id: str) -> str:
    """Return this month's consumption for a meter ID."""
    return {"MTR-8801": "412 kWh, 3% above last month",
            "MTR-9042": "180 kWh, unchanged"}.get(meter_id, "no reading")


@mcp.resource("meter://{meter_id}")
def meter_info(meter_id: str) -> str:
    """Metadata for one meter."""
    return f"Station {meter_id}, automatic, hourly readings"


if __name__ == "__main__":
    mcp.run(transport="stdio")

### 2. Call it in process

`Client` accepts a server object directly, which is the quickest way to try a server
without spawning anything.

In [ ]:
import asyncio
from meter_server import mcp
from mcp import Client


async def demo():
    async with Client(mcp) as client:
        listed = await client.list_tools()
        for t in listed.tools:
            print(t.name, "->", t.description)
        print()
        print("schema:", listed.tools[0].input_schema)

        result = await client.call_tool("find_meter", {"city": "Kerkstraat 12"})
        print("content   :", result.content[0].text)
        print("structured:", result.structured_content)


await demo()

### 3. What actually goes over the wire

Run the same server as a subprocess and speak JSON-RPC to it by hand.

Notice there is no handshake. The first message is a real request, because since
2026-07-28 MCP is stateless and every request carries its own `_meta`.

In [ ]:
import json, subprocess, sys

META = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    "io.modelcontextprotocol/clientCapabilities": {},
    "io.modelcontextprotocol/clientInfo": {"name": "course-demo", "version": "1.0.0"},
}

proc = subprocess.Popen([sys.executable, "meter_server.py"],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                        stderr=subprocess.DEVNULL, text=True, bufsize=1)


def rpc(method, params=None):
    req = {"jsonrpc": "2.0", "id": rpc.n, "method": method,
           "params": {**(params or {}), "_meta": META}}
    rpc.n += 1
    proc.stdin.write(json.dumps(req) + chr(10))
    proc.stdin.flush()
    return json.loads(proc.stdout.readline())


rpc.n = 1
print(json.dumps(rpc("tools/call", {"name": "find_meter",
                                    "arguments": {"city": "Kerkstraat 12"}}), indent=2))

A tool that fails is not a JSON-RPC error. It returns a normal result with `isError` set, so the model can read the message and try something else.

In [ ]:
print(json.dumps(rpc("tools/call", {"name": "no_such_tool", "arguments": {}}), indent=2))
proc.terminate()

### 4. Hand the tools to an agent

MCP tools are not LangChain tools, so they need wrapping. This is the whole bridge:
list the tools, wrap each one as a `StructuredTool` whose coroutine calls back through
the MCP client.

In [ ]:
from langchain_core.tools import StructuredTool


def to_langchain_tools(client, mcp_tools):
    """Wrap each MCP tool as a LangChain StructuredTool."""
    tools = []
    for t in mcp_tools:
        async def call(_name=t.name, **kwargs):
            result = await client.call_tool(_name, kwargs)
            return result.content[0].text

        tools.append(StructuredTool.from_function(
            coroutine=call,
            name=t.name,
            description=t.description,
            args_schema=t.input_schema,
        ))
    return tools

In [ ]:
from langchain.agents import create_agent


async def run_agent(question):
    async with Client(mcp) as client:
        listed = await client.list_tools()
        agent = create_agent(llm, to_langchain_tools(client, listed.tools))
        result = await agent.ainvoke({"messages": [("human", question)]})
        return result["messages"][-1].content


print(await run_agent("How much did Kerkstraat 12 use?"))

The agent had to chain two tools: `find_meter` to turn the city into an ID, then
`read_meter` to read that ID. The order was not specified anywhere ; the model derived it from the two
docstrings. It worked it out
from the two docstrings, which is why those docstrings are part of the program.

### 5. The same server over HTTP

Only the transport changes. Run this in a terminal:

```bash
python meter_server.py --transport streamable-http
```

then point a client at the URL instead of the object:

```python
async with Client("http://localhost:8000/mcp") as client:
    ...
```

That is the same server our agent just used, now reachable by any MCP client,
including Claude and the OpenAI Agents SDK.